# Feature Engineering on Full Data
**by Group 3**

Install matplotlib and scipy libraries

In [1]:
!pip install matplotlib
!pip install scipy

  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.3.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.8 MB)
Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (325 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x

Java is installed for PySpark

In [2]:
!sudo yum install -y java-11-amazon-corretto-headless

Last metadata expiration check: 0:19:42 ago on Sun Jul 26 07:54:50 2026.
Dependencies resolved.
 Package                       Arch   Version                 Repository   Size
Installing:
 java-11-amazon-corretto-headless
                               x86_64 1:11.0.31+11-1.amzn2023 amazonlinux  91 M

Transaction Summary
Install  1 Package

Total download size: 91 M
Installed size: 226 M
java-11-amazon-corretto-headless-11.0.31+11-1.a  67 MB/s |  91 MB     00:01    
--------------------------------------------------------------------------------
Total                                            65 MB/s |  91 MB     00:01     
Running transaction check
Transaction check succeeded.
Running transaction test
Transaction test succeeded.
Running transaction
  Preparing        :                                                        1/1 
  Installing       : java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn   1/1 
  Running scriptlet: java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn   1

In [3]:
import os
import glob

# Search for the installed Java directory
java_paths = glob.glob('/usr/lib/jvm/java-11*')

if java_paths:
    # Dynamically set the environment variable to the found path
    os.environ["JAVA_HOME"] = java_paths[0]
    print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Did the yum install command work?")

JAVA_HOME successfully set to: /usr/lib/jvm/java-11-amazon-corretto.x86_64


Importing the necessary libraries for feature engineering

In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import time

Setup a SparkSession

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, input_file_name, countDistinct, desc, round, avg
from pyspark.sql.functions import date_format
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("DAT204M-EDA-Sagemaker")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "10g")
    .getOrCreate()
)

INPUT_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
OUTPUT_PATH = "s3a://dat204m-project-g3/sampled_eda_data/"


print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-268266d2-1964-448a-be94-c5fb824a6297;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (94ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (1237ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.

26/07/26 08:15:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark ready: 3.3.0


Read the clean data from S3

In [6]:
DATA_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
df = spark.read.parquet(DATA_PATH)

26/07/26 08:15:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Enables eager evaluation in Spark so DataFrames are automatically displayed in a formatted HTML table when referenced in notebook cells, making data inspection easier without explicitly calling .show().

In [7]:
# Enable clean HTML formatting for Spark DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

Import necessary PySpark libraries

In [8]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark import StorageLevel

Aggregates the raw event-level dataset into a product-level feature table by grouping records by product_id. For each product, it computes customer engagement metrics (views, likes, cart additions, offers, purchases), counts unique users and sessions, calculates the average price, extracts representative product metadata (brand and category path), summarizes item condition counts, and replaces missing values with default values to produce a complete feature set for clustering.

In [9]:
features_df = (
    df.groupBy("product_id")
      .agg(
          # Event counts
          F.sum(F.when(F.col("event_id") == "item_view", 1).otherwise(0)).alias("views"),
          F.sum(F.when(F.col("event_id") == "item_like", 1).otherwise(0)).alias("likes"),
          F.sum(F.when(F.col("event_id") == "item_add_to_cart_tap", 1).otherwise(0)).alias("cart"),
          F.sum(F.when(F.col("event_id") == "offer_make", 1).otherwise(0)).alias("offers"),
          F.sum(F.when(F.col("event_id") == "buy_start", 1).otherwise(0)).alias("buy_start"),
          F.sum(F.when(F.col("event_id") == "buy_comp", 1).otherwise(0)).alias("buy_comp"),

          # Engagement
          F.countDistinct("user_id").alias("unique_users"),
          F.countDistinct("session_id").alias("unique_sessions"),

          # Price
          F.coalesce(F.avg("price"), F.lit(0.0)).alias("avg_price"),

          # Metadata
          F.coalesce(
              F.first("brand_name", ignorenulls=True),
              F.lit("Unknown")
          ).alias("brand_name"),

          # Category path
          F.concat_ws(
              " > ",
              F.coalesce(F.first("c0_name", ignorenulls=True), F.lit("Unknown")),
              F.coalesce(F.first("c1_name", ignorenulls=True), F.lit("Unknown")),
              F.coalesce(F.first("c2_name", ignorenulls=True), F.lit("Unknown"))
          ).alias("category_path"),

          # Condition
          F.sum(F.when(F.col("item_condition_name") == "Good", 1).otherwise(0)).alias("cond_good"),
          F.sum(F.when(F.col("item_condition_name") == "New", 1).otherwise(0)).alias("cond_new"),
          F.sum(F.when(F.col("item_condition_name") == "Like new", 1).otherwise(0)).alias("cond_like_new"),
          F.sum(F.when(F.col("item_condition_name") == "Fair", 1).otherwise(0)).alias("cond_fair"),
          F.sum(F.when(F.col("item_condition_name") == "Poor", 1).otherwise(0)).alias("cond_poor"),
          F.sum(
                F.when(
                    F.col("item_condition_name").isNull() |
                    (F.trim(F.col("item_condition_name")) == ""),
                    1
                ).otherwise(0)
            ).alias("cond_unknown")
      )
)

# Final safety net
features_df = features_df.fillna({
    "views": 0,
    "likes": 0,
    "cart": 0,
    "offers": 0,
    "buy_start": 0,
    "buy_comp": 0,
    "unique_users": 0,
    "unique_sessions": 0,
    "avg_price": 0.0,
    "brand_name": "Unknown",
    "category_path": "Unknown",
    "cond_good": 0,
    "cond_new": 0,
    "cond_like_new": 0,
    "cond_fair": 0,
    "cond_poor": 0,
    "cond_unknown": 0
})

In [ ]:
print(features_df.show(10))

[Stage 2:>                (4 + 4) / 690][Stage 3:>                (0 + 0) / 690]

log1p() was applied to highly skewed numerical features (e.g., views, likes, users, sessions, and price) to reduce the influence of extreme values and compress the range of the data. This helps K-Means form more balanced clusters, as the algorithm is sensitive to large differences in feature magnitudes. The log1p() function (log(1+x)) is used instead of log(x) because it safely handles zero values.

In [10]:
features_df = (
    features_df
    .withColumn("log_views", F.log1p("views"))
    .withColumn("log_likes", F.log1p("likes"))
    .withColumn("log_cart", F.log1p("cart"))
    .withColumn("log_offers", F.log1p("offers"))
    .withColumn("log_buy_start", F.log1p("buy_start"))
    .withColumn("log_buy_comp", F.log1p("buy_comp"))
    .withColumn("log_users", F.log1p("unique_users"))
    .withColumn("log_sessions", F.log1p("unique_sessions"))
    .withColumn("log_price", F.log1p("avg_price"))
    .withColumn("log_cond_good", F.log1p("cond_good"))
    .withColumn("log_cond_new", F.log1p("cond_new"))
    .withColumn("log_cond_like_new", F.log1p("cond_like_new"))
    .withColumn("log_cond_fair", F.log1p("cond_fair"))
    .withColumn("log_cond_poor", F.log1p("cond_poor"))
    .withColumn("log_cond_unknown", F.log1p("cond_unknown"))
)

For model training, the data is saved to S3

In [29]:
s3_path = "s3a://dat204m-project-g3/feature_engineered_full"

features_df.write \
    .mode("overwrite") \
    .parquet(s3_path)